In [1]:
import cobra

# user inputs
model_file = '/data2/hratch/human_me/prebuild/recon2_2.xml'
exclude_small_molecules = True
membrane_diffusion_limit = 504
# 504 Da includes ATP (500 according to https://pubs.acs.org/doi/pdf/10.1021/acs.jmedchem.7b00237)
#, uncharged molecules at this diffusion limit are passive, no dummy

In [2]:
def flatten_list(t):
    #https://stackoverflow.com/questions/952914/how-to-make-a-flat-list-out-of-list-of-lists
    return [item for sublist in t for item in sublist]

In [4]:
model = cobra.io.read_sbml_model(model_file)

In [140]:
def determine_transport(r):
    '''
    Parameters
    ----------
    r: cobra.core.reactions.Reaction

    Returns
    ----------
    actual_transport_m: list
        a list of metabolites that are actually transported across compartments
        each element is a string of the metabolite id without the compartment ('_compartment' ending)
    
    
    '''
    
    sm_reactants = dict()
    sm_prod = dict()
    for m in r.reactants:
        m_id = m.id.split('_')[:-1][0]
        if m_id not in sm_reactants:
            sm_reactants[m_id] = [m.id.split('_')[-1]]
        else: 
            sm_reactants[m_id] += [m.id.split('_')[-1]]
    for m in r.products:
        m_id = m.id.split('_')[:-1][0]
        if m_id not in sm_prod:
            sm_prod[m_id] = [m.id.split('_')[-1]]
        else: 
            sm_prod[m_id] += [m.id.split('_')[-1]]
    potential_transport_m = set(sm_prod).intersection(sm_reactants)
    actual_transport_m = [m for m in potential_transport_m if len(set(sm_reactants[m]).intersection(sm_prod[m])) == 0  
                         and (sm_reactants[m] + sm_prod[m] != ['e', 'b'])] # this accounts for LParen_EParen reactions
    return actual_transport_m

In [154]:
# get enzymeless reactions
enzymeless_reactions = [r.id for r in model.reactions if len(r.genes) == 0]

#---------
# get transport reactions 
transport_reactions = [r.id for r in model.reactions if len(r.compartments)>1]                       

# double check that a metabolite is being transported
exclude = list()
for r_id in transport_reactions:
    actual_transport_m = determine_transport(model.reactions.get_by_id(r_id))
    if len(actual_transport_m) == 0:
        exclude.append(r_id)
transport_reactions = list(set(transport_reactions).difference(exclude))
                      
#---------

#get enzymeless transpor reactions
transport_reactions = [model.reactions.get_by_id(r_id) for r_id in set(transport_reactions).intersection(enzymeless_reactions)]

In [155]:
if exclude_small_molecules:
    print('Before filtering, there are {} transport reactions'.format(len(transport_reactions)))
    exclude = []
    for r in transport_reactions:
        # filter for molecules that are actually transported (in reactants and products) and check 
        # if they are over the diffusion limit or charged
        actual_transport_m = determine_transport(r)
        sm_transport = [m for m in r.reactants if (m.id.split('_')[:-1][0] in actual_transport_m) and \
                           ((m.formula_weight is not None and m.formula_weight > membrane_diffusion_limit) \
                            or (m.charge != 0))]
        if len(sm_transport) == 0:
            exclude.append(r.id)
            
    transport_reactions = [r for r in transport_reactions if r.id not in exclude]
    print('After filtering, there are {} transport reactions'.format(len(transport_reactions)))

Before filtering, there are 1303 transport reactions
After filtering, there are 866 transport reactions
